# B3 — LightGBM, MLP & Đặc trưng đồ thị trên Kaggle (Chạy OFFLINE — Không cần bật Internet)

1. **Add Input** → thêm dataset *IBM Transactions for Anti Money Laundering (AML)* (`ealtman2019`).
2. **Add Input → Upload → New Dataset** → tải lên file `NCKH2026_b3_offline.zip` (tên bất kỳ, vd `nckh2026-b3`).
3. **Không cần bật Internet**, không cần xác minh SĐT. Chọn **Run All** (hoặc **Save Version → Save & Run All**).
4. Xong → tải `/kaggle/working/ket_qua.zip` ở tab **Output**.

In [ ]:
import glob, shutil
from pathlib import Path

hits = sorted(glob.glob('/kaggle/input/**/scripts/b3_lightgbm.py', recursive=True))
assert hits, 'Chưa thấy dataset code NCKH2026! Hãy Upload NCKH2026_b3_offline.zip vào mục Input.'
SRC_REPO = Path(hits[0]).parents[1]
DST_REPO = Path('/kaggle/working/NCKH2026')
if DST_REPO.exists():
    shutil.rmtree(DST_REPO)
shutil.copytree(SRC_REPO, DST_REPO)
%cd /kaggle/working/NCKH2026
!ls -la

In [ ]:
# Chạy test kiểm tra trực tiếp trên các thư viện có sẵn của Kaggle (không cần cài gì thêm)
!python -m pytest -q tests/test_dac_trung_do_thi.py tests/test_b3_lightgbm.py tests/test_khung.py tests/test_ibm_features.py tests/test_metrics.py tests/test_sampling.py tests/test_ibm_aml.py -p no:cacheprovider

In [ ]:
import glob
RAW = sorted(glob.glob('/kaggle/input/**/HI-Small_Trans.csv', recursive=True))[0]
ACC = sorted(glob.glob('/kaggle/input/**/HI-Small_accounts.csv', recursive=True))[0]
print(RAW, ACC, sep='\n')

In [ ]:
# Tính 22 đặc trưng cơ bản một lần (~3-5 phút)
!python scripts/06_ibm_features.py --input {RAW} --view bank

In [ ]:
# Kiểm tra B0: phải ra đủ 3 dòng KHỚP trước khi chạy thí nghiệm
!python scripts/b0_chia_co_dinh.py --kiem-tra 2>&1 | tee results/kaggle_b0_kiem_tra.txt

In [ ]:
# ===== Chạy thí nghiệm B3 (LightGBM + đặc trưng đồ thị) =====
# Mặc định: Chạy thử trên fold 0, seed 42 (~5-8 phút tổng cộng)
!python scripts/b3_lightgbm.py --raw {RAW} --mo-hinh lgbm --bo-dac-trung co_ban do_thi_fl do_thi_nuoc do_thi_global --folds 0 --seeds 42 --tag thu_b3_lgbm_f0

# Chạy đủ 5 fold cho LightGBM (bỏ dấu # nếu muốn chạy luôn):
# !python scripts/b3_lightgbm.py --raw {RAW} --mo-hinh lgbm --bo-dac-trung co_ban do_thi_fl do_thi_nuoc do_thi_global --folds 0 1 2 3 4 --seeds 42 --tag b3_lgbm_5fold_s42
# Chạy thêm MLP (sau khi đã xem xong kết quả LightGBM):
# !python scripts/b3_lightgbm.py --raw {RAW} --mo-hinh mlp --bo-dac-trung co_ban do_thi_fl do_thi_nuoc do_thi_global --folds 0 --seeds 42 --tag thu_b3_mlp_f0

In [ ]:
!cd /kaggle/working/NCKH2026 && rm -f /kaggle/working/ket_qua.zip && zip -rq /kaggle/working/ket_qua.zip results -x "results/_luu_tru/*" && unzip -l /kaggle/working/ket_qua.zip | tail -n 20